# 03 · Agregações de Negócio 

🎯 **Objetivo:** Consolidar o uso de `groupBy`/`agg`/`orderBy` para responder perguntas de negócio com dados agregados.

**Teoria:** docs/04-dataframes-catalyst-tungsten.md

Ainda `local[*]`. Este notebook fica só em `vendas` — sem joins — para você se concentrar no padrão **agrupar → agregar → ordenar** antes de complicar com múltiplas tabelas (isso vem no notebook 04).

---
### 📊 O que são agregações?

Agregações transformam **muitas linhas** em **poucas linhas de resumo**. Exemplos do dia a dia:
- Total de vendas por mês
- Média de salário por cargo
- Contagem de funcionários por empresa

No Spark, o padrão é sempre: **groupBy → agg → orderBy**.

### 🔤 Operações que você vai praticar

1. **Agrupamento** — `groupBy`
2. **Agregação** — `agg` com `sum`, `avg`, `count`, `min`, `max`
3. **Ordenação** — `orderBy` (já visto no notebook 02)
4. **Filtro pós-agregação** — `filter`/`where` sobre o resultado de um `agg` (o equivalente ao `HAVING` do SQL)
5. **Agregação customizada** — `pandas_udf` do tipo Series-to-scalar dentro de `groupBy().agg()`

Vamos praticar!


In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("app-01")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.executor.memory", "2g")
    .config("spark.executor.cores", "2")
    .getOrCreate()
)

spark

In [ ]:
# Lê cada tabela da camada Bronze em formato Parquet (colunar, comprimido)
# Criar Spark Data Frame
sdf_empresas = spark.read.parquet("../data/bronze/empresas")
sdf_funcionarios = spark.read.parquet("../data/bronze/funcionarios")
sdf_vendas = spark.read.parquet("../data/bronze/vendas")

✅ **Dados carregados.** A tabela `vendas` contém transações com valor, data (ano/mês/dia) e referências aos funcionários. Vamos explorar esses dados com agregações.


## Total de vendas por ano/mês

Como a receita evoluiu mês a mês? Esta é a pergunta clássica de **série temporal** que o `groupBy` resolve.

🧠 **Padrão:** `groupBy(colunas_agrupadoras).agg(função_agregadora)`. O Spark:
1. Embaralha os dados para reunir linhas do mesmo grupo (shuffle)
2. Aplica a função de agregação dentro de cada grupo
3. Retorna um DataFrame com uma linha por grupo

In [ ]:
from pyspark.sql.functions import col, sum as spark_sum
# Importa a função sum com alias para não conflitar com sum() do Python

# Agrupa por ano e mês, soma o valor das vendas, ordena cronologicamente
vendas_por_periodo = (
    sdf_vendas.groupBy("ano", "mes")
    .agg(spark_sum("valor").alias("total_vendas"))
    .orderBy("ano", "mes")
)
# Exibe todos os 24 meses (2 anos de dados)
vendas_por_periodo.show(24)

📌 **Interpretação da saída:**

- Cada linha representa um mês de vendas.
- A coluna `total_vendas` é a soma de todas as vendas daquele período.
- A ordenação por ano/mês revela a **sazonalidade** do negócio.

💡 **Dica:** Repare que usamos `spark_sum("valor")` e não `sum("valor")`. Isso porque o Python já tem uma função `sum()` embutida. O alias evita conflito.


## Ticket médio e volume de transações

Duas métricas fundamentais para qualquer negócio:
- **Ticket médio:** qual o valor típico de cada venda?
- **Volume:** quantas transações aconteceram no total?

💡 **Dica:** `avg()` e `count()` são funções de agregação embutidas no Spark SQL. Quando usadas sem `groupBy`, elas agregam **todas as linhas** em uma única linha de resultado.

In [ ]:
from pyspark.sql.functions import avg, count

# Agregação global (sem groupBy) — uma única linha de resultado
# count("*") conta todas as linhas, incluindo nulos
sdf_vendas.agg(
    avg("valor").alias("ticket_medio"),
    count("*").alias("total_transacoes"),
).show()

📌 **O que o ticket médio nos diz?**

- Um ticket médio baixo pode indicar vendas de volume alto (muitas vendas de pouco valor).
- Um ticket médio alto pode indicar vendas mais esporádicas mas de maior valor.
- O `count(*)` conta todas as linhas — incluindo valores nulos em qualquer coluna.

🧠 **Desafio:** Como você calcularia o ticket médio **por ano**?


## As 10 maiores vendas individuais

Ao contrário das agregações anteriores, aqui **não agrupamos** — apenas ordenamos todas as linhas pela coluna `valor` em ordem decrescente e exibimos as 10 primeiras.

🧠 **Por quê?** `orderBy().show(10)` com `desc()` é um padrão muito usado para encontrar outliers, registros suspeitos ou apenas conhecer os extremos dos dados.

In [ ]:
# Projeta as colunas relevantes e ordena por valor decrescente
maiores_vendas = (
    sdf_vendas.select("id_venda", "id_funcionario", "valor", "ano", "mes", "dia")
    .orderBy(col("valor").desc())
)
# Exibe apenas as 10 maiores — não carrega todo o dataset na memória
maiores_vendas.show(10)

📌 **Análise:** As 10 maiores vendas revelam os valores máximos do dataset. Compare com o ticket médio — a diferença mostra a **dispersão** dos dados.

⚠️ **Atenção:** `orderBy().show(10)` é eficiente porque o Spark usa uma otimização chamada **TakeOrderedAndProject** — ele encontra os 10 maiores sem precisar ordenar o dataset inteiro.


## Vendas por ano: total, contagem, ticket médio, mínimo e máximo

✅ **Resolvendo o desafio anterior:** para calcular o ticket médio **por ano**, basta adicionar `"ano"` ao `groupBy` — é exatamente isso que fazemos abaixo.

`agg` aceita **várias agregações de uma vez** — não precisa de uma chamada por métrica. Isso é mais eficiente porque o Spark processa todas as agregações em uma única passada sobre os dados.

💡 **Dica:** Use `alias()` para dar nomes legíveis às colunas agregadas. Caso contrário, o Spark usa nomes genéricos como `sum(valor)`.

In [ ]:
from pyspark.sql.functions import min as spark_min, max as spark_max

# Múltiplas agregações em um único groupBy — processamento eficiente
resumo_anual = (
    sdf_vendas.groupBy("ano")
    .agg(
        spark_sum("valor").alias("total_vendas"),
        count("*").alias("numero_vendas"),
        avg("valor").alias("ticket_medio"),
        spark_min("valor").alias("menor_venda"),
        spark_max("valor").alias("maior_venda"),
    )
    .orderBy("ano")
)
resumo_anual.show()

📌 **Vantagem de múltiplas agregações:**

- Em vez de rodar 5 consultas separadas (uma para total, uma para contagem, uma para média, uma para mínimo, uma para máximo), fazemos tudo em **um único `agg`**.
- O Spark Catalyst Optimizer consegue fundir as operações em um único estágio de shuffle.
- Resultado: uma tabela resumo por ano com todas as métricas lado a lado — repare como `menor_venda` e `maior_venda` batem com o que já vimos na seção "10 maiores vendas".

💡 **Dica:** Além de `sum`, `avg`, `count`, `min` e `max`, existem dezenas de outras funções — `stddev()`, `approx_count_distinct()`, `skewness()`, `kurtosis()` — todas utilizáveis dentro do `agg`.


#### 💡 **Exemplo 1:** Mediana e média de vendas por ano com `pandas_udf`

Assim como no notebook 02, `avg`/`mean` não é a única forma de resumir uma distribuição — e o Spark não tem `median()` nativo. Reaproveitamos a mesma pandas UDF Series-to-scalar de lá, agora dentro de `groupBy().agg()`, para calcular a mediana **e** a média **por grupo**, usando duas pandas UDFs na mesma chamada de `agg`.

⚠️ **Limitação:** pandas UDFs de agregação (Series-to-scalar) podem ser combinadas **entre si** na mesma chamada de `agg` — como fazemos abaixo com `mediana()` e `media()`. O que **não** funciona é misturar uma pandas UDF de agregação com uma função de agregação **nativa** do Spark (`sum()`, `avg()`, `count()`...) na mesma chamada — isso dispara `[INVALID_PANDAS_UDF_PLACEMENT]`. Por isso usamos nossa própria `media()` em pandas, em vez do `avg()` nativo do Spark.

In [ ]:
import pandas as pd
from pyspark.sql.functions import pandas_udf


# Duas pandas UDFs Series-to-scalar — a mesma mediana() do notebook 02, e uma nova media()
@pandas_udf("double")
def mediana(valores: pd.Series) -> float:
    return valores.median()


@pandas_udf("double")
def media(valores: pd.Series) -> float:
    return valores.mean()


# Duas pandas UDFs de agregação PODEM ser combinadas na mesma chamada de agg()
vendas_mediana_por_ano = (
    sdf_vendas.groupBy("ano")
    .agg(
        mediana(col("valor")).alias("mediana_vendas"),
        media(col("valor")).alias("media_vendas"),
    )
    .orderBy("ano")
)
vendas_mediana_por_ano.show()

📌 **Comparando mediana e média:** a `mediana_vendas` (~R\$54,5 a R\$54,8) fica bem abaixo da `media_vendas` (~R\$90) em todos os anos — o mesmo padrão de distribuição assimétrica (muitas vendas baixas, poucas vendas altas) que vimos no notebook 02, agora com as duas métricas lado a lado na mesma tabela (e batendo com o `ticket_medio` de `resumo_anual`, calculado com `avg()` nativo).

#### 💡 **Exemplo 2:** Meses com vendas acima da média — o padrão `HAVING`

SQL tem a cláusula `HAVING` para filtrar **depois** de agregar (diferente de `WHERE`, que filtra **antes**). No Spark não existe um método `.having()` — o padrão é: agregue, guarde o resultado como um novo DataFrame e aplique `filter()` sobre ele, exatamente como fizemos no Exemplo 4 do notebook 02 com a média salarial.

In [ ]:
# Média do total_vendas mensal — calculada sobre o resultado JÁ agregado de vendas_por_periodo
media_mensal = vendas_por_periodo.agg(avg("total_vendas")).first()[0]
print(f"Média mensal de vendas: R$ {media_mensal:,.2f}")

# "HAVING total_vendas > média" — filter() aplicado sobre o DataFrame já agregado
meses_acima_da_media = (
    vendas_por_periodo
    .filter(col("total_vendas") > media_mensal)
    .orderBy(col("total_vendas").desc())
)
meses_acima_da_media.show(30)
print(f"{meses_acima_da_media.count()} de {vendas_por_periodo.count()} meses ficaram acima da média.")

📌 **Entendendo a saída:** os meses de 2026 dominam o topo da lista — sinal de que o negócio está **crescendo**: o total mensal de vendas em 2026 supera consistentemente a média histórica calculada sobre os três anos completos.

#### 💡 **Exemplo 3:** Ranking dos 5 funcionários que mais venderam (sem join)

A tabela `vendas` já guarda `id_funcionario` — não precisamos abrir a tabela `funcionarios` para descobrir **quanto** cada um vendeu, só para descobrir o **nome** de quem vendeu mais. `groupBy` + `agg` + `orderBy` + `limit` já respondem à pergunta de negócio; os nomes ficam para o próximo notebook, quando juntamos as tabelas com `join`.

In [ ]:
top_vendedores = (
    sdf_vendas.groupBy("id_funcionario")
    .agg(
        spark_sum("valor").alias("total_vendido"),
        count("*").alias("numero_vendas"),
    )
    .orderBy(col("total_vendido").desc())
    .limit(5)
)
top_vendedores.show()

📌 **O que falta aqui?** Só o `id_funcionario` — sabemos **quanto** cada um vendeu, mas não **quem** é. Essa é exatamente a pergunta que o `join` (próximo notebook) resolve: cruzar `vendas` com `funcionarios` para trazer o nome, cargo e empresa de cada top vendedor.

In [ ]:
# Encerra a SparkSession — libera threads e memória
spark.stop()

---
🎉 **Agregações concluídas!** Você aprendeu:

- `groupBy` para agrupar dados por colunas categóricas
- `agg` com `sum`, `avg`, `count`, `min`, `max` para calcular métricas
- `orderBy` com `desc()` para ordenar resultados
- Múltiplas agregações em uma única passada
- `pandas_udf` (Series-to-scalar) como agregação customizada dentro de `groupBy().agg()`
- O padrão `HAVING`: filtrar **depois** de agregar, aplicando `filter()` sobre o resultado do `agg`
- Ranking de negócio com `groupBy` + `agg` + `orderBy` + `limit` — sem precisar de `join`

▶️ **Próximo:** Notebook 04 — Joins entre Vendas, Funcionários e Empresas
